In [28]:
#%pip install numpy pandas matplotlib statsmodels
#%pip install EMD-signal
#%pip install scikit-learn
#%pip install --upgrade plotly nbformat

In [13]:
#import numpy as np
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from PyEMD import EMD  # pip install PyEMD
from sklearn.preprocessing import MinMaxScaler

# STL-EMD based DATC Functions

In [52]:
def estimate_period_fft(series, default_period=24):
    """
    FFT를 활용하여 시계열 데이터에서 가장 지배적인 주기를 동적으로 추정합니다.
    """
    n = len(series)
    t = np.arange(n)
    
    # 1. FFT 전처리: 선형 추세 및 평균(DC 성분) 제거
    p = np.polyfit(t, series, 1)
    detrended = series - np.polyval(p, t)
    detrended = detrended - np.mean(detrended)
    
    # 2. FFT 수행 및 진폭 계산
    fft_vals = np.fft.rfft(detrended)
    frequencies = np.fft.rfftfreq(n)
    magnitudes = np.abs(fft_vals)
    
    # 3. 0Hz(상수항)를 제외하고 진폭이 가장 큰 인덱스 탐색
    if len(magnitudes) > 1:
        dominant_idx = np.argmax(magnitudes[1:]) + 1 
        dominant_freq = frequencies[dominant_idx]
        
        if dominant_freq > 0:
            period = int(np.round(1.0 / dominant_freq))
            if 2 <= period <= n // 2:
                return period
                
    return default_period

def calculate_time_series_strength(series):
    """
    1. 고전적 STL 분해: y = T + S + R_stl -> F_T_STL, F_S_STL 산출
    2. STL-EMD 하이브리드 분해: y = T + S + I + R_pure -> F_T_STL_EMD, F_S_STL_EMD, F_I_STL_EMD 산출
    """
    estimated_p = estimate_period_fft(series)
    
    if len(series) < 2 * estimated_p:
        estimated_p = max(2, len(series) // 2 - 1)
        
    try:
        # [1단계] 고전적 STL 분해 수행 (y = T + S + R_stl)
        res = STL(series, period=estimated_p, robust=True).fit()
        T, S, R_stl = res.trend, res.seasonal, res.resid
        
        var_R_stl = np.var(R_stl)
        var_TR_stl = np.var(T + R_stl)
        var_SR_stl = np.var(S + R_stl)
        
        # STL 기반 피처 연산
        F_T_STL = max(0.0, 1.0 - (var_R_stl / var_TR_stl if var_TR_stl > 0 else 1.0))
        F_S_STL = max(0.0, 1.0 - (var_R_stl / var_SR_stl if var_SR_stl > 0 else 1.0))
        
        # [2단계] EMD 분해 기동 (R_stl을 I + R_pure로 분리)
        emd = EMD()
        imfs = emd.emd(R_stl)
        
        if len(imfs) > 2:
            I = np.sum(imfs[1:], axis=0) 
        elif len(imfs) == 2:
            I = imfs[1]
        else:
            I = np.zeros_like(R_stl)
            
        # 순수 가우시안 확률 노이즈 정제 (y = T + S + I + R_pure)
        R_pure = R_stl - I
        
        # 최종 정제된 R_pure 기반의 강도 재연산
        var_R_pure = np.var(R_pure)
        var_TR_pure = np.var(T + R_pure) 
        var_SR_pure = np.var(S + R_pure)
        var_IR_pure = np.var(I + R_pure) 
        
        # STL-EMD 기반 피처 연산
        F_T_STL_EMD = max(0.0, 1.0 - (var_R_pure / var_TR_pure if var_TR_pure > 0 else 1.0))
        F_S_STL_EMD = max(0.0, 1.0 - (var_R_pure / var_SR_pure if var_SR_pure > 0 else 1.0))
        F_I_STL_EMD = max(0.0, 1.0 - (var_R_pure / var_IR_pure if var_IR_pure > 0 else 1.0))
            
    except Exception:
        # 수리적 예외 발생 시 0.0 반환
        F_T_STL, F_S_STL = 0.0, 0.0
        F_T_STL_EMD, F_S_STL_EMD, F_I_STL_EMD = 0.0, 0.0, 0.0
        
    return F_T_STL, F_S_STL, F_T_STL_EMD, F_S_STL_EMD, F_I_STL_EMD, estimated_p


def generate_synthetic_dataset(num_samples_per_regime=50, len_series=500):
    """
    모든 시계열 샘플에 대해 STL 피처와 STL-EMD 피처를 모두 산출하여 기록하는 데이터셋 생성기
    """
    print(f"🔹 Generating synthetic dataset (Samples/Regime: {num_samples_per_regime}, Length: {len_series})")

    data_records = []
    t = np.arange(len_series)
    
    for i in range(num_samples_per_regime):
        print(f"Generating sample {i+1}/{num_samples_per_regime} for each regime...")
        rand_p1 = np.random.choice([12, 24, 36, 48])
        rand_p2 = rand_p1 * 2
        
        # --- R1: Composite Regime (High Trend + High Seasonality) ---
        slope = np.random.uniform(0.1, 0.4) * np.random.choice([1, -1])
        curve = np.random.uniform(1e-5, 5e-5) * np.random.choice([1, -1])
        amp1, amp2 = np.random.uniform(8, 25), np.random.uniform(2, 8)
        noise_std = np.random.uniform(0.5, 1.5)
        
        q1_series = (slope * t + curve * (t**2)) + (amp1 * np.sin(2 * np.pi * t / rand_p1)) + (amp2 * np.cos(2 * np.pi * t / rand_p2)) + np.random.normal(0, noise_std, len_series)
        q1_scaled = (q1_series - np.min(q1_series)) / (np.max(q1_series) - np.min(q1_series) + 1e-9)
        ft_stl, fs_stl, ft_stl_emd, fs_stl_emd, fi_stl_emd, p_est = calculate_time_series_strength(q1_scaled)
        
        data_records.append({
            'True_Regime': 'R1', 'Pattern': 'Composite', 
            'F_T_STL': ft_stl, 'F_S_STL': fs_stl, 
            'F_T_STL_EMD': ft_stl_emd, 'F_S_STL_EMD': fs_stl_emd, 'F_I_STL_EMD': fi_stl_emd, 
            'True_P': rand_p1, 'Est_P': p_est,
            'rand_p1': rand_p1, 'rand_p2': rand_p2, 'length': len_series, 'slope': slope, 'curve': curve,
            'amp1': amp1, 'amp2': amp2, 'noise_std': noise_std, 'phi': None, 'break_point': None, 'drift': None
        })
        
        # --- R2: Pure Seasonal (Low Trend + High Seasonality) ---
        amp1, amp2 = np.random.uniform(10, 30), np.random.uniform(4, 12)
        noise_std = np.random.uniform(0.5, 2.0)
        
        q2_series = (amp1 * np.sin(2 * np.pi * t / rand_p1)) + (amp2 * np.cos(2 * np.pi * t / rand_p2)) + np.random.normal(0, noise_std, len_series)
        q2_scaled = (q2_series - np.min(q2_series)) / (np.max(q2_series) - np.min(q2_series) + 1e-9)
        ft_stl, fs_stl, ft_stl_emd, fs_stl_emd, fi_stl_emd, p_est = calculate_time_series_strength(q2_scaled)
        
        data_records.append({
            'True_Regime': 'R2', 'Pattern': 'Seasonal', 
            'F_T_STL': ft_stl, 'F_S_STL': fs_stl, 
            'F_T_STL_EMD': ft_stl_emd, 'F_S_STL_EMD': fs_stl_emd, 'F_I_STL_EMD': fi_stl_emd, 
            'True_P': rand_p1, 'Est_P': p_est,
            'rand_p1': rand_p1, 'rand_p2': rand_p2, 'length': len_series, 'slope': 0.0, 'curve': 0.0,
            'amp1': amp1, 'amp2': amp2, 'noise_std': noise_std, 'phi': None, 'break_point': None, 'drift': None
        })

        # --- R3: Stationary / Noise (Low Trend + Low Seasonality) ---
        noise_type = np.random.choice(['WN', 'AR'])
        phi_val = None
        
        if noise_type == 'WN':
            noise_std_wn = np.random.uniform(5, 15)
            q3_series = np.random.normal(0, noise_std_wn, len_series)
            current_noise_std = noise_std_wn
        else:
            phi_val = np.random.uniform(0.4, 0.7)
            q3_series = np.zeros(len_series)
            innovations_std = 5.0
            innovations = np.random.normal(0, innovations_std, len_series)
            q3_series[0] = innovations[0]
            for idx in range(1, len_series):
                q3_series[idx] = phi_val * q3_series[idx-1] + innovations[idx]
            current_noise_std = innovations_std

        q3_scaled = (q3_series - np.min(q3_series)) / (np.max(q3_series) - np.min(q3_series) + 1e-9)
        ft_stl, fs_stl, ft_stl_emd, fs_stl_emd, fi_stl_emd, p_est = calculate_time_series_strength(q3_scaled)
        
        data_records.append({
            'True_Regime': 'R3', 'Pattern': f'Stationary_{noise_type}', 
            'F_T_STL': ft_stl, 'F_S_STL': fs_stl, 
            'F_T_STL_EMD': ft_stl_emd, 'F_S_STL_EMD': fs_stl_emd, 'F_I_STL_EMD': fi_stl_emd, 
            'True_P': 'None', 'Est_P': p_est,
            'rand_p1': None, 'rand_p2': None, 'length': len_series, 'slope': 0.0, 'curve': 0.0, 
            'amp1': 0.0, 'amp2': 0.0, 'noise_std': current_noise_std, 'phi': phi_val, 'break_point': None, 'drift': None
        })

        # --- R4: Pure Trending with Mean Shift (High Trend + Low Seasonality) ---
        trend_type = np.random.choice(['MeanShift', 'RandomWalkDrift', 'ExponentialGrowth'])
        noise_std = np.random.uniform(0.5, 1.5)
        bp_val = None
        drift_val = None
        
        if trend_type == 'MeanShift':
            bp_val = len_series // 2
            q4_series = np.random.normal(0, noise_std, len_series)
            q4_series[bp_val:] += np.random.uniform(50, 150)
        elif trend_type == 'RandomWalkDrift':
            drift_val = np.random.uniform(0.1, 0.5) * np.random.choice([1, -1])
            q4_series = np.cumsum(np.random.normal(drift_val, 2, len_series))
        else:
            q4_series = np.exp(np.linspace(0, np.random.uniform(3, 5), len_series)) + np.random.normal(0, noise_std, len_series)

        q4_scaled = (q4_series - np.min(q4_series)) / (np.max(q4_series) - np.min(q4_series) + 1e-9)
        ft_stl, fs_stl, ft_stl_emd, fs_stl_emd, fi_stl_emd, p_est = calculate_time_series_strength(q4_scaled)
        
        data_records.append({
            'True_Regime': 'R4', 'Pattern': f'Trending_{trend_type}', 
            'F_T_STL': ft_stl, 'F_S_STL': fs_stl, 
            'F_T_STL_EMD': ft_stl_emd, 'F_S_STL_EMD': fs_stl_emd, 'F_I_STL_EMD': fi_stl_emd, 
            'True_P': 'None', 'Est_P': p_est,
            'rand_p1': None, 'rand_p2': None, 'length': len_series, 'slope': None, 'curve': None, 
            'amp1': 0.0, 'amp2': 0.0, 'noise_std': noise_std, 'phi': None, 'break_point': bp_val, 'drift': drift_val
        })
        
    return pd.DataFrame(data_records)

In [53]:
#df_exp_results = generate_synthetic_dataset(num_samples_per_regime=1000, len_series=500, dec_method='stl')
df_exp_results = generate_synthetic_dataset(num_samples_per_regime=1000, len_series=500)
df_exp_results.to_csv('../../results/temp/0714_Per1000_Len500_synthetic_dataset.csv', index=False)

🔹 Generating synthetic dataset (Samples/Regime: 1000, Length: 500)
Generating sample 1/1000 for each regime...
Generating sample 2/1000 for each regime...
Generating sample 3/1000 for each regime...
Generating sample 4/1000 for each regime...
Generating sample 5/1000 for each regime...
Generating sample 6/1000 for each regime...
Generating sample 7/1000 for each regime...
Generating sample 8/1000 for each regime...
Generating sample 9/1000 for each regime...
Generating sample 10/1000 for each regime...
Generating sample 11/1000 for each regime...
Generating sample 12/1000 for each regime...
Generating sample 13/1000 for each regime...
Generating sample 14/1000 for each regime...
Generating sample 15/1000 for each regime...
Generating sample 16/1000 for each regime...
Generating sample 17/1000 for each regime...
Generating sample 18/1000 for each regime...
Generating sample 19/1000 for each regime...
Generating sample 20/1000 for each regime...
Generating sample 21/1000 for each regime.

In [64]:
df_exp_results = df_exp_results.fillna(0)
#df_exp_results[df_exp_results['Pattern'].str.contains('Trending')].head(10)
df_exp_results.head(20)

,True_Regime,Pattern,F_T_STL,F_S_STL,F_T_STL_EMD,F_S_STL_EMD,F_I_STL_EMD,True_P,Est_P,rand_p1,rand_p2,length,slope,curve,amp1,amp2,noise_std,phi,break_point,drift
0,R1,Composite,0.992535,0.964532,0.998191,0.991279,0.761699,12,12,12.0,24.0,500,0.125865,0.000041,13.501126,4.103770,1.417998,0.000000,0.0,0.000000
1,R2,Seasonal,0.715801,0.987284,0.894574,0.998699,0.898592,12,12,12.0,24.0,500,0.000000,0.000000,22.434386,4.622837,0.868605,0.000000,0.0,0.000000
2,R3,Stationary_WN,0.241310,0.197423,0.415513,0.277953,0.298334,None,3,0.0,0.0,500,0.000000,0.000000,0.000000,0.000000,12.594962,0.000000,0.0,0.000000
3,R4,Trending_RandomWalkDrift,0.879061,0.598939,0.975598,0.944473,0.819352,None,167,0.0,0.0,500,0.000000,0.000000,0.000000,0.000000,1.403952,0.000000,0.0,0.208746
4,R1,Composite,0.993285,0.976918,0.999607,0.998644,0.942489,12,12,12.0,24.0,500,-0.222951,0.000050,21.803272,6.463595,0.796401,0.000000,0.0,0.000000
5,R2,Seasonal,0.736214,0.978614,0.952052,0.998944,0.951471,12,12,12.0,24.0,500,0.000000,0.000000,17.891930,5.143974,0.631891,0.000000,0.0,0.000000
6,R3,Stationary_WN,0.069106,0.221877,0.079891,0.328134,0.319333,None,14,0.0,0.0,500,0.000000,0.000000,0.000000,0.000000,13.679280,0.000000,0.0,0.000000
7,R4,Trending_ExponentialGrowth,0.998846,0.248430,0.999279,0.410413,0.369174,None,24,0.0,0.0,500,0.000000,0.000000,0.000000,0.000000,0.999038,0.000000,0.0,0.000000
8,R1,Composite,0.998838,0.973466,0.999718,0.993342,0.757014,12,12,12.0,24.0,500,-0.398649,0.000050,15.719321,3.921039,1.340592,0.000000,0.0,0.000000
9,R2,Seasonal,0.726681,0.916628,0.905218,0.991356,0.903885,12,12,12.0,24.0,500,0.000000,0.000000,11.799418,6.755743,1.105405,0.000000,0.0,0.000000


In [113]:
import plotly.graph_objects as go
import pandas as pd

# 임곗값(Threshold) 상수 정의
F_T_thr = 0.8
F_S_thr = 0.64
F_I_thr = 0.45

# 축의 전체 렌더링 범위 정의
axis_min, axis_max = -0.05, 1.05

# 1. 3D 지원 규격에 맞춰 마커의 내부를 모두 색상으로 채우는(Solid) 설정 테이블 선언
pattern_configs = {
    'Composite':                  {'color': '#d63031', 'symbol': 'circle', 'label': 'R1: Composite'},
    
    'Seasonal':                   {'color': '#0984e3', 'symbol': 'circle', 'label': 'R2: Seasonal'},
    
    'Stationary_WN':              {'color': '#009432', 'symbol': 'square', 'label': 'R3: Stationary-WN'},
    'Stationary_AR':              {'color': '#009432', 'symbol': 'x', 'label': 'R3: Stationary-AR'},
    
    'Trending_MeanShift':         {'color': '#8e44ad', 'symbol': 'diamond', 'label': 'R4: Trending-Mean Shift'},
    'Trending_RandomWalkDrift':   {'color': '#8e44ad', 'symbol': 'cross', 'label': 'R4: Trending-RW Drift'},
    'Trending_ExponentialGrowth': {'color': '#8e44ad', 'symbol': 'diamond', 'label': 'R4: Trending-Exp Growth'}
}

# 범례 고정 정렬 순서
legend_order = [
    'Composite', 'Seasonal', 'Stationary_WN', 'Stationary_AR',
    'Trending_MeanShift', 'Trending_RandomWalkDrift', 'Trending_ExponentialGrowth'
]

# 2. Plotly 3D Figure 객체 초기화
fig = go.Figure()

# 3. 고정된 범례 순서에 따라 속이 가득 찬 데이터 트레이스(Trace) 주입
for pattern_label in legend_order:
    if pattern_label not in df_exp_results['Pattern'].unique():
        continue
        
    group = df_exp_results[df_exp_results['Pattern'] == pattern_label]
    config = pattern_configs[pattern_label]
    
    fig.add_trace(go.Scatter3d(
        x=group['F_T_STL'],
        y=group['F_S_STL'],
        z=group['F_I_STL_EMD'],
        mode='markers',
        name=config['label'],
        marker=dict(
            size=6,
            color=config['color'],
            symbol=config['symbol'],
            opacity=0.90,
            line=dict(width=0.3, color='rgba(255,255,255,0.8)')
        ),
        text=group['Pattern'],
        hovertemplate=(
            "<b>Pattern: %{text}</b><br>" +
            "F_T_STL: %{x:.3f}<br>" +
            "F_S_STL: %{y:.3f}<br>" +
            "F_I_STL_EMD: %{z:.3f}<extra></extra>"
        )
    ))

# 4. 임곗값(Threshold) 기준선 추가 (3D 공간 내 점선 배치)
# (1) Trend Threshold Line (X = 0.8)
fig.add_trace(go.Scatter3d(
    x=[F_T_thr, F_T_thr], y=[axis_min, axis_max], z=[0, 0],
    mode='lines',
    name='F_T Threshold (0.8)',
    line=dict(color='black', width=4, dash='dash'),
    showlegend=True
))

# (2) Seasonal Threshold Line (Y = 0.64)
fig.add_trace(go.Scatter3d(
    x=[axis_min, axis_max], y=[F_S_thr, F_S_thr], z=[0, 0],
    mode='lines',
    name='F_S Threshold (0.64)',
    line=dict(color='black', width=4, dash='dash'),
    showlegend=True
))

# (3) Intervention Threshold Line (Z = 0.45)
# 기준 평면의 원점(0,0)에서 Z축 위로 솟아오르는 기준선을 정의합니다.
fig.add_trace(go.Scatter3d(
    x=[0, 0], y=[axis_min, axis_max], z=[F_I_thr, F_I_thr],
    mode='lines',
    name='F_I Threshold (0.45)',
    line=dict(color='darkred', width=4, dash='dash'),
    showlegend=True
))

# 5. 레이아웃 뷰포트 정돈 및 테마 구성
fig.update_layout(
    title='<b>Interactive 3D Feature Space: Threshold & Solid Marker Exploration</b>',
    title_x=0.5,
    margin=dict(l=0, r=0, b=0, t=60),
    template='plotly_white',
    scene=dict(
        xaxis=dict(title='Trend Strength (F_T_STL)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        yaxis=dict(title='Seasonal Strength (F_S_STL)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        zaxis=dict(title='Intervention Strength (F_I_STL_EMD)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.3)
        )
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5,
        font=dict(size=10.5),
        bordercolor="gray",
        borderwidth=1
    )
)

# 6. 기본 웹 브라우저 창으로 화면 강제 출력
fig.show(renderer="browser")

In [115]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

# 1. 임곗값(Threshold) 상수 정의
F_T_thr = 0.8
F_S_thr = 0.64
F_I_thr = 0.45

# 축의 전체 렌더링 범위 정의
axis_min, axis_max = -0.05, 1.05

# 2. 평면(Plane) 격자 데이터 생성을 위한 좌표 배열 준비
grid_resolution = 10
u = np.linspace(axis_min, axis_max, grid_resolution)
v = np.linspace(axis_min, axis_max, grid_resolution)
U, V = np.meshgrid(u, v)

# 3. 고유 시각 세팅 테이블 선언 (Solid 마커 규격)
pattern_configs = {
    'Composite':                  {'color': '#d63031', 'symbol': 'circle', 'label': 'R1: Composite'},

    'Seasonal':                   {'color': '#0984e3', 'symbol': 'circle', 'label': 'R2: Seasonal'},

    'Stationary_WN':              {'color': '#009432', 'symbol': 'circle', 'label': 'R3: Stationary-WN'},
    'Stationary_AR':              {'color': '#009432', 'symbol': 'x', 'label': 'R3: Stationary-AR'},

    'Trending_MeanShift':         {'color': '#8e44ad', 'symbol': 'circle', 'label': 'R4: Trending-Mean Shift'},
    'Trending_RandomWalkDrift':   {'color': '#8e44ad', 'symbol': 'x', 'label': 'R4: Trending-RW Drift'},
    'Trending_ExponentialGrowth': {'color': '#8e44ad', 'symbol': 'diamond', 'label': 'R4: Trending-Exp Growth'}
}

# 범례 고정 정렬 순서
legend_order = [
    'Composite', 'Seasonal', 'Stationary_WN', 'Stationary_AR',
    'Trending_MeanShift', 'Trending_RandomWalkDrift', 'Trending_ExponentialGrowth'
]

# 4. Plotly 3D Figure 객체 초기화
fig = go.Figure()

# 5. 데이터 트레이스(Trace) 주입 (모든 패턴을 기본 True로 설정)
for pattern_label in legend_order:
    if pattern_label not in df_exp_results['Pattern'].unique():
        continue
        
    group = df_exp_results[df_exp_results['Pattern'] == pattern_label]
    config = pattern_configs[pattern_label]
    
    fig.add_trace(go.Scatter3d(
        x=group['F_T_STL'],
        y=group['F_S_STL'],
        z=group['F_I_STL_EMD'],
        mode='markers',
        name=config['label'],
        # [핵심] visible=True를 명시하여 모든 Regime이 처음 실행 시 즉각 보이도록 설정합니다.
        # 이 상태에서 범례를 클릭하면 Plotly 엔진이 알아서 'invisible'로 토글합니다.
        visible=True, 
        marker=dict(
            size=6,
            color=config['color'],
            symbol=config['symbol'],
            opacity=0.90,
            # 흰색 테두리 중첩으로 하얗게 타는 현상을 방지하기 위해 테두리를 제거하거나 최소화
            line=dict(width=0, color='rgba(0,0,0,0)') 
        ),
        text=group['Pattern'],
        hovertemplate=(
            "<b>Pattern: %{text}</b><br>" +
            "F_T_STL: %{x:.3f}<br>" +
            "F_S_STL: %{y:.3f}<br>" +
            "F_I_STL_EMD: %{z:.3f}<extra></extra>"
        )
    ))

# 6. 임곗값 기준 3D 평면(Surface Planes) 추가
# (1) F_T Threshold Plane (X = 0.8)
fig.add_trace(go.Surface(
    x=np.full_like(U, F_T_thr), y=U, z=V,
    colorscale=[[0, 'rgba(128,128,128,0.10)'], [1, 'rgba(128,128,128,0.10)']],
    showscale=False,
    name=f'F_T Gate Plane ({F_T_thr})',
    showlegend=True
))

# (2) F_S Threshold Plane (Y = 0.64)
fig.add_trace(go.Surface(
    x=U, y=np.full_like(V, F_S_thr), z=V,
    colorscale=[[0, 'rgba(128,128,128,0.10)'], [1, 'rgba(128,128,128,0.10)']],
    showscale=False,
    name=f'F_S Gate Plane ({F_S_thr})',
    showlegend=True
))

# (3) F_I Threshold Plane (Z = 0.45)
fig.add_trace(go.Surface(
    x=U, y=V, z=np.full_like(U, F_I_thr),
    colorscale=[[0, 'rgba(128,128,128,0.20)'], [1, 'rgba(128,128,128,0.20)']],
    showscale=False,
    name=f'F_I Gate Plane ({F_I_thr})',
    showlegend=True
))

# 7. 레이아웃 뷰포트 정돈 및 테마 구성
fig.update_layout(
    #title='<b>Interactive 3D Feature Space: Calibrated 3D Threshold Planes</b>',
    #title_x=0.5,
    #margin=dict(l=0, r=0, b=0, t=60),
    template='plotly_white',
    scene=dict(
        xaxis=dict(title='Trend Strength (F_T_STL)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        yaxis=dict(title='Seasonal Strength (F_S_STL)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        zaxis=dict(title='Intervention Strength (F_I_STL_EMD)', range=[axis_min, axis_max], gridcolor='rgba(0,0,0,0.1)'),
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.3)
        )
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5,
        font=dict(size=10.5),
        bordercolor="gray",
        borderwidth=1
    )
)

# 8. 기본 웹 브라우저 창으로 화면 강제 출력
fig.show(renderer="browser")

In [ ]:
import pandas as pd
import numpy as np

# 1. 임곗값(Threshold) 상수 정의
F_T_thr = 0.8
F_S_thr = 0.64
F_I_thr = 0.45

# 2. Regime 한글 명칭 정의 (출력용)
regime_names = {
    'R1': 'Composite (복합 패턴 영역)',
    'R2': 'Seasonal (계절성 패턴 영역)',
    'R3': 'Stationary (정상성 노이즈 영역)',
    'R4': 'Trending (추세성 패턴 영역)'
}

# 변수 초기화
total_success = 0

print("=" * 60)
print("  [3D Feature Space Regime Mapping Verification Result]  ")
print("=" * 60)

# 3. 각 True Regime 그룹별 매핑 루프 돌기
for r_label in ['R1', 'R2', 'R3', 'R4']:
    # True_Regime 기준으로 서브 데이터프레임 분리
    sub_df = df_exp_results[df_exp_results['True_Regime'] == r_label]
    
    if len(sub_df) == 0:
        continue

    # [핵심] 사용자가 지정한 3차원 공간 게이트 분할 조건문 구현
    # 데이터셋의 실제 Intervention 강도 컬럼명은 'F_I_STL_EMD'입니다.
    if r_label == 'R1':
        success_condition = (
            (sub_df['F_T_STL'] >= F_T_thr) & 
            (sub_df['F_S_STL'] >= F_S_thr) & 
            (sub_df['F_I_STL_EMD'] >= F_I_thr)
        )
    elif r_label == 'R2':
        success_condition = (
            (sub_df['F_T_STL'] < F_T_thr) & 
            (sub_df['F_S_STL'] >= F_S_thr)
        )
    elif r_label == 'R3':
        success_condition = (
            (sub_df['F_T_STL'] < F_T_thr) & 
            (sub_df['F_S_STL'] < F_S_thr)
        )
    elif r_label == 'R4':
        # R4의 이중 영역 조건 결합: 수평 분할 평면(F_I) 하단 영역 + 상단 영역의 R4 지분
        cond_1 = (sub_df['F_T_STL'] >= F_T_thr) & (sub_df['F_S_STL'] < F_S_thr)
        cond_2 = (sub_df['F_T_STL'] >= F_T_thr) & (sub_df['F_S_STL'] >= F_S_thr) & (sub_df['F_I_STL_EMD'] < F_I_thr)
        success_condition = cond_1 | cond_2
        
    # 성공 데이터 개수 및 정확도 산출
    success_count = len(sub_df[success_condition])
    total_success += success_count
    accuracy = (success_count / len(sub_df)) * 100
    
    # 기초 메타데이터 추출
    unique_patterns = list(sub_df['Pattern'].unique())
    
    # 결과 출력
    print(f"▶ {r_label}: {regime_names[r_label]}")
    print(f"  - 포함된 패턴 종류: {unique_patterns} (총 {len(unique_patterns)}개)")
    print(f"  - 평균 특징 강도  : Mean F_T = {sub_df['F_T_STL'].mean():.3f} | Mean F_S = {sub_df['F_S_STL'].mean():.3f} | Mean F_I = {sub_df['F_I_STL_EMD'].mean():.3f}")

    # 주기 매칭 검증 (FFT 주기 추정 성과는 주기 성분이 지支配적인 R1, R2만 출력)
    if r_label in ['R1', 'R2'] and 'True_P' in sub_df.columns and 'Est_P' in sub_df.columns:
        same_p_count = len(sub_df[sub_df['True_P'] == sub_df['Est_P']])
        p_match_rate = (same_p_count / len(sub_df)) * 100
        print(f"  - FFT 주기 추정 정확도: {p_match_rate:.1f}%")
        
    print(f"  - 사분면 매핑 성공률 : [ {success_count} / {len(sub_df)} ] ({accuracy:.1f}%)")
    print("-" * 60)

# 4. 전체 데이터 기준 통합 정확도 출력
print(f"■ 전체 통합 오라클 라우팅 정확도: {(total_success / len(df_exp_results))*100:.2f}%")
print("=" * 60)

  [3D Feature Space Regime Mapping Verification Result]  
▶ R1: Composite (복합 패턴 영역)
  - 포함된 패턴 종류: ['Composite'] (총 1개)
  - 평균 특징 강도  : Mean F_T = 0.994 | Mean F_S = 0.959 | Mean F_I = 0.868
  - FFT 주기 추정 정확도: 74.3%
  - 사분면 매핑 성공률 : [ 999 / 1000 ] (99.9%)
------------------------------------------------------------
▶ R2: Seasonal (계절성 패턴 영역)
  - 포함된 패턴 종류: ['Seasonal'] (총 1개)
  - 평균 특징 강도  : Mean F_T = 0.640 | Mean F_S = 0.943 | Mean F_I = 0.898
  - FFT 주기 추정 정확도: 72.7%
  - 사분면 매핑 성공률 : [ 1000 / 1000 ] (100.0%)
------------------------------------------------------------
▶ R3: Stationary (정상성 노이즈 영역)
  - 포함된 패턴 종류: ['Stationary_WN', 'Stationary_AR'] (총 2개)
  - 평균 특징 강도  : Mean F_T = 0.134 | Mean F_S = 0.239 | Mean F_I = 0.431
  - 사분면 매핑 성공률 : [ 980 / 1000 ] (98.0%)
------------------------------------------------------------
▶ R4: Trending (추세성 패턴 영역)
  - 포함된 패턴 종류: ['Trending_RandomWalkDrift', 'Trending_ExponentialGrowth', 'Trending_MeanShift'] (총 3개)
  - 평균 특징 강도  : Mean F_T = 0.988